# Analyse exploratoire des données (EDA) — AdventureWorks

**Problématique :** Comment la performance des fournisseurs et la rotation des stocks influencent-elles la marge commerciale, de l'achat jusqu'à la vente ?

**Plan du notebook :**
0. Configuration
1. Chargement et diagnostic basique des 15 tables
2. Vérification des clés primaires
3. Intégrité référentielle (clés étrangères)
4. Cohérence logique (dates, valeurs négatives)
5. Diagnostics par axe : fournisseurs, stocks, marge
6. Analyse approfondie : prix d'achat réel vs prix standard (`ProductVendor`)
7. Synthèse

## 0. Configuration

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 150)

# Localisation du dossier clean_data, que le notebook soit à la racine du projet ou dans un sous-dossier
BASE = Path.cwd() if (Path.cwd() / "clean_data").exists() else Path.cwd().parent
DOSSIER = BASE / "clean_data"
print(f"Dossier des données : {DOSSIER}")

Dossier des données : /tmp/claude-0/-home-claude/10ed7d5f-04c5-514b-9736-c3b923fc50b6/scratchpad/work/clean_data


## 1. Chargement et diagnostic basique

Pour chaque table : dimensions, valeurs manquantes, doublons (lignes 100 % identiques) et types de données.

In [2]:
def diagnostic_basique(nom_fichier, df):
    print(f"\n{'='*60}")
    print(f"📋 {nom_fichier}")
    print(f"{'='*60}")
    print(f"Dimensions : {df.shape[0]} lignes, {df.shape[1]} colonnes")

    # Valeurs manquantes
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) > 0:
        pct = (nulls / len(df) * 100).round(1)
        print(f"\n⚠️ Valeurs manquantes :")
        for col in nulls.index:
            print(f"   {col} : {nulls[col]} ({pct[col]}%)")
    else:
        print("\n✅ Aucune valeur manquante")

    # Lignes dupliquées (toutes colonnes identiques)
    doublons = df.duplicated().sum()
    if doublons > 0:
        print(f"\n⚠️ Lignes dupliquées (toutes colonnes) : {doublons}")
    else:
        print("✅ Aucun doublon parfait (toutes colonnes)")

    # Types de données
    print(f"\nTypes de données :")
    print(df.dtypes.to_string())

In [3]:
fichiers = [
    "Vendor", "ProductVendor", "PurchaseOrderHeader", "PurchaseOrderDetail",
    "ShipMethod", "Location", "Product", "ProductInventory",
    "ProductCategory", "ProductSubcategory", "Customer", "Person",
    "SalesOrderHeader", "SalesOrderDetail", "SalesTerritory"
]

dfs = {}
for f in fichiers:
    try:
        dfs[f] = pd.read_csv(DOSSIER / f"{f}.csv", sep=";")
        diagnostic_basique(f, dfs[f])
    except FileNotFoundError:
        print(f"❌ Fichier introuvable : {DOSSIER / f'{f}.csv'}")


📋 Vendor
Dimensions : 104 lignes, 8 colonnes

⚠️ Valeurs manquantes :
   purchasingwebserviceurl : 98 (94.2%)
✅ Aucun doublon parfait (toutes colonnes)

Types de données :
businessentityid           int64
accountnumber                str
name                         str
creditrating               int64
preferredvendorstatus      int64
activeflag                 int64
purchasingwebserviceurl      str
modifieddate                 str

📋 ProductVendor
Dimensions : 460 lignes, 11 colonnes

⚠️ Valeurs manquantes :
   onorderqty : 305 (66.3%)
✅ Aucun doublon parfait (toutes colonnes)

Types de données :
productid             int64
businessentityid      int64
averageleadtime       int64
standardprice       float64
lastreceiptcost     float64
lastreceiptdate         str
minorderqty           int64
maxorderqty           int64
onorderqty          float64
unitmeasurecode         str
modifieddate            str

📋 PurchaseOrderHeader
Dimensions : 4012 lignes, 13 colonnes

✅ Aucune valeur manquant


📋 Customer
Dimensions : 19820 lignes, 7 colonnes

⚠️ Valeurs manquantes :
   personid : 701 (3.5%)
   storeid : 18484 (93.3%)
✅ Aucun doublon parfait (toutes colonnes)

Types de données :
customerid         int64
personid         float64
storeid          float64
territoryid        int64
accountnumber        str
rowguid              str
modifieddate         str



📋 Person
Dimensions : 19972 lignes, 13 colonnes

⚠️ Valeurs manquantes :
   title : 18963 (94.9%)
   middlename : 8499 (42.6%)
   suffix : 19919 (99.7%)
   additionalcontactinfo : 19962 (99.9%)
✅ Aucun doublon parfait (toutes colonnes)

Types de données :
businessentityid         int64
persontype                 str
namestyle                int64
title                      str
firstname                  str
middlename                 str
lastname                   str
suffix                     str
emailpromotion           int64
additionalcontactinfo      str
demographics               str
rowguid                    str
modifieddate               str



📋 SalesOrderHeader
Dimensions : 31465 lignes, 26 colonnes

⚠️ Valeurs manquantes :
   purchaseordernumber : 27659 (87.9%)
   salespersonid : 27659 (87.9%)
   creditcardid : 1131 (3.6%)
   creditcardapprovalcode : 1131 (3.6%)
   currencyrateid : 17489 (55.6%)
   comment : 31465 (100.0%)
✅ Aucun doublon parfait (toutes colonnes)

Types de données :
salesorderid                int64
revisionnumber              int64
orderdate                     str
duedate                       str
shipdate                      str
status                      int64
onlineorderflag             int64
salesordernumber              str
purchaseordernumber           str
accountnumber                 str
customerid                  int64
salespersonid             float64
territoryid                 int64
billtoaddressid             int64
shiptoaddressid             int64
shipmethodid                int64
creditcardid              float64
creditcardapprovalcode        str
currencyrateid            float64
subt


📋 SalesOrderDetail
Dimensions : 121317 lignes, 11 colonnes

⚠️ Valeurs manquantes :
   carriertrackingnumber : 60398 (49.8%)
✅ Aucun doublon parfait (toutes colonnes)

Types de données :
salesorderid               int64
salesorderdetailid         int64
carriertrackingnumber        str
orderqty                   int64
productid                  int64
specialofferid             int64
unitprice                float64
unitpricediscount        float64
linetotal                float64
rowguid                      str
modifieddate                 str

📋 SalesTerritory
Dimensions : 10 lignes, 10 colonnes

✅ Aucune valeur manquante
✅ Aucun doublon parfait (toutes colonnes)

Types de données :
territoryid            int64
name                     str
countryregioncode        str
group                    str
salesytd             float64
saleslastyear        float64
costytd              float64
costlastyear         float64
rowguid                  str
modifieddate             str


### Constat — section 1

- **15 tables chargées**, aucune ligne dupliquée à l'identique.
- Les valeurs manquantes sont majoritairement **structurelles** (liées au modèle métier) et non des erreurs de saisie ; elles sont conservées pour ne pas introduire de biais.
- Colonnes entièrement vides (`Product.discontinueddate`, `SalesOrderHeader.comment`) : aucune information, à ignorer dans l'analyse.

#### Inventaire complet des valeurs manquantes (23 colonnes)

| Table | Colonne | % manquant | Cause métier | Traitement |
|---|---|---|---|---|
| Vendor | `purchasingwebserviceurl` | 94,2 % | Fournisseur sans service web d'achat | Conservé NULL (hors périmètre) |
| ProductVendor | `onorderqty` | 66,3 % | Aucune commande en cours | Imputé 0 |
| Product | `color`, `size`, `style`, `class`, `productline` | 44,8 % – 58,1 % | Composants et accessoires sans attribut d'apparence | Conservé NULL (non applicable) |
| Product | `sizeunitmeasurecode` | 65,1 % | Pas de taille → pas d'unité | Conservé NULL |
| Product | `weight`, `weightunitmeasurecode` | 59,3 % | Produit non pesé | Conservé NULL (non utilisé) |
| Product | `productsubcategoryid`, `productmodelid` | 41,5 % | Composants non vendables | Conservé NULL |
| Product | `sellenddate` | 80,6 % | Produit toujours en vente | Conservé NULL + indicateur `is_active` |
| Product | `discontinueddate` | **100 %** | Aucun produit arrêté | Colonne vide → supprimable |
| ProductInventory | `shelf` | 27,1 % | Emplacement non renseigné | Imputé « N/A » |
| Customer | `personid` / `storeid` | 3,5 % / 93,3 % | Type de client (particulier / magasin) | Conservé NULL (exclusion mutuelle) |
| Person | `title`, `suffix` | 94,9 % / 99,7 % | Champs facultatifs | Conservé NULL (hors périmètre) |
| Person | `middlename` | 42,6 % | Absence de second prénom | Conservé NULL |
| Person | `additionalcontactinfo` | 99,9 % | Champ facultatif | Conservé NULL (hors périmètre) |
| SalesOrderHeader | `salespersonid`, `purchaseordernumber` | 87,9 % | Commandes en ligne (sans commercial) | Conservé NULL |
| SalesOrderHeader | `currencyrateid` | 55,6 % | Commandes en USD | Conservé NULL |
| SalesOrderHeader | `creditcardid`, `creditcardapprovalcode` | 3,6 % | Autre mode de paiement | Conservé NULL |
| SalesOrderHeader | `comment` | **100 %** | Jamais renseigné | Colonne vide → supprimable |
| SalesOrderDetail | `carriertrackingnumber` | 49,8 % | Commandes en ligne sans numéro de suivi transporteur | Conservé NULL |

In [4]:
soh = dfs["SalesOrderHeader"]
print(pd.crosstab(soh["onlineorderflag"], soh["purchaseordernumber"].isnull(),
                  rownames=["onlineorderflag"], colnames=["PO manquant"]))

PO manquant      False  True 
onlineorderflag              
0                 3806      0
1                    0  27659


| SalesOrderHeader | `salespersonid`, `purchaseordernumber` | 87,9 % | Commandes en ligne de particuliers (B2C) : ni commercial, ni bon de commande client — vérifié : 100 % des NULL correspondent à `onlineorderflag = 1` | Conservé NULL |

**Constat :** la correspondance est parfaite : les 27 659 commandes en ligne (`onlineorderflag = 1`) n'ont jamais de bon de commande client, et les 3 806 commandes revendeurs (`onlineorderflag = 0`) en ont toujours un. Les valeurs manquantes sont donc **structurelles** et non des erreurs de saisie.

#### Valeurs manquantes « déguisées » (zéros)

`isnull()` ne détecte que les NULL. Dans `Product`, de nombreux produits ont `standardcost = 0` et `listprice = 0` : un zéro qui signifie en réalité « non applicable ». Ces valeurs faussent les moyennes et les calculs de marge si elles ne sont pas identifiées.

In [5]:
p = dfs["Product"]

for col in ["standardcost", "listprice"]:
    nb_zero = (p[col] == 0).sum()
    print(f"{col} = 0 : {nb_zero} produits ({nb_zero / len(p):.1%})")

# Ces produits sont-ils des composants (non vendables) ?
print("\nRépartition des produits à listprice = 0 :")
print(p[p["listprice"] == 0][["makeflag", "finishedgoodsflag"]].value_counts())

# Contrôle : un produit vendable avec un prix à 0 serait une vraie anomalie
anomalies = p[(p["finishedgoodsflag"] == 1) & ((p["listprice"] == 0) | (p["standardcost"] == 0))]
print(f"\n⚠️ Produits finis (vendables) avec un prix ou coût à 0 : {len(anomalies)}")

standardcost = 0 : 200 produits (39.7%)
listprice = 0 : 200 produits (39.7%)

Répartition des produits à listprice = 0 :
makeflag  finishedgoodsflag
0         0                    182
1         0                     18
Name: count, dtype: int64

⚠️ Produits finis (vendables) avec un prix ou coût à 0 : 0


**Constat :**
- **200 produits (39,7 %)** ont `standardcost = 0` et `listprice = 0` : ce sont les mêmes produits.
- Ils sont **tous non vendables** (`finishedgoodsflag = 0`) : 182 composants achetés (`makeflag = 0`) et 18 sous-ensembles fabriqués (`makeflag = 1`).
- **Aucun produit fini** n'a un prix ou un coût à 0 → pas d'anomalie de saisie.

**Décision :** pour ces composants, le zéro signifie « non applicable » (pas de prix de vente catalogue). Ils sont exclus des indicateurs de prix et de marge.

Après l'import en base, le script `nettoyage.py` applique un traitement supplémentaire directement dans SQLite (les fichiers Parquet exportés par `export_clean_parquet.py` en héritent) : suppression des colonnes techniques ou vides (`rowguid`, `modifieddate`, `comment`, `discontinueddate`, `purchasingwebserviceurl`, `title`, `suffix`, `additionalcontactinfo`), `onorderqty` NULL → 0, `carriertrackingnumber` NULL → « Unknown », et imputation des attributs textuels de Product (color, size, style, class, productline, unités de mesure) par « Unknown », afin d'éviter les valeurs vides dans les filtres et segments Power BI. Les colonnes numériques (weight) et les clés (productsubcategoryid, productmodelid) restent NULL.

## 2. Vérification des clés primaires

`duplicated()` ne détecte que les lignes entièrement identiques. Ici, on vérifie pour **les 15 tables** que la clé primaire (simple ou composite) respecte les deux conditions : **non nulle** et **unique**.

In [6]:
def verifier_cle_primaire(df, cols_pk, nom_table):
    # Colonnes absentes : on signale au lieu de lever une KeyError
    manquantes = [c for c in cols_pk if c not in df.columns]
    if manquantes:
        print(f"❌ {nom_table} : colonne(s) introuvable(s) {manquantes}")
        return

    # Une clé primaire doit être (1) non nulle et (2) unique
    nulls = df[cols_pk].isnull().any(axis=1).sum()
    doublons = df.duplicated(subset=cols_pk).sum()

    if nulls == 0 and doublons == 0:
        print(f"✅ {nom_table} : clé primaire {cols_pk} unique et non nulle")
    else:
        if nulls > 0:
            print(f"⚠️ {nom_table} : {nulls} valeurs nulles dans la clé primaire {cols_pk}")
        if doublons > 0:
            print(f"⚠️ {nom_table} : {doublons} doublons sur la clé primaire {cols_pk}")

In [7]:
# Clé primaire de chaque table (simple ou composite)
cles_primaires = {
    "Vendor":              ["businessentityid"],
    "ProductVendor":       ["productid", "businessentityid"],
    "PurchaseOrderHeader": ["purchaseorderid"],
    "PurchaseOrderDetail": ["purchaseorderid", "purchaseorderdetailid"],
    "ShipMethod":          ["shipmethodid"],
    "Location":            ["locationid"],
    "Product":             ["productid"],
    "ProductInventory":    ["productid", "locationid"],
    "ProductCategory":     ["productcategoryid"],
    "ProductSubcategory":  ["productsubcategoryid"],
    "Customer":            ["customerid"],
    "Person":              ["businessentityid"],
    "SalesOrderHeader":    ["salesorderid"],
    "SalesOrderDetail":    ["salesorderid", "salesorderdetailid"],
    "SalesTerritory":      ["territoryid"],
}

for table, cols in cles_primaires.items():
    if table in dfs:
        verifier_cle_primaire(dfs[table], cols, table)

✅ Vendor : clé primaire ['businessentityid'] unique et non nulle
✅ ProductVendor : clé primaire ['productid', 'businessentityid'] unique et non nulle
✅ PurchaseOrderHeader : clé primaire ['purchaseorderid'] unique et non nulle
✅ PurchaseOrderDetail : clé primaire ['purchaseorderid', 'purchaseorderdetailid'] unique et non nulle
✅ ShipMethod : clé primaire ['shipmethodid'] unique et non nulle
✅ Location : clé primaire ['locationid'] unique et non nulle
✅ Product : clé primaire ['productid'] unique et non nulle
✅ ProductInventory : clé primaire ['productid', 'locationid'] unique et non nulle
✅ ProductCategory : clé primaire ['productcategoryid'] unique et non nulle
✅ ProductSubcategory : clé primaire ['productsubcategoryid'] unique et non nulle
✅ Customer : clé primaire ['customerid'] unique et non nulle
✅ Person : clé primaire ['businessentityid'] unique et non nulle
✅ SalesOrderHeader : clé primaire ['salesorderid'] unique et non nulle
✅ SalesOrderDetail : clé primaire ['salesorderid', 

**Constat :** les 15 clés primaires (11 simples, 4 composites) sont **uniques et non nulles**. Aucune correction nécessaire avant l'import en base.

## 3. Intégrité référentielle (clés étrangères)

Pour chaque relation, on recherche les valeurs « orphelines » : présentes dans la table enfant mais absentes de la table parente.

In [8]:
def verifier_cle_etrangere(df_enfant, col_enfant, df_parent, col_parent, nom_relation):
    ids_enfant = set(df_enfant[col_enfant].dropna())
    ids_parent = set(df_parent[col_parent].dropna())
    orphelins = ids_enfant - ids_parent
    if orphelins:
        print(f"⚠️ {nom_relation} : {len(orphelins)} valeurs orphelines "
              f"(présentes dans l'enfant, absentes du parent)")
        print(f"   Exemples : {list(orphelins)[:5]}")
    else:
        print(f"✅ {nom_relation} : intégrité référentielle OK")

In [9]:
verifier_cle_etrangere(dfs["PurchaseOrderDetail"], "productid", dfs["Product"], "productid",
                       "PurchaseOrderDetail.productid → Product.productid")
verifier_cle_etrangere(dfs["SalesOrderDetail"], "productid", dfs["Product"], "productid",
                       "SalesOrderDetail.productid → Product.productid")
verifier_cle_etrangere(dfs["PurchaseOrderHeader"], "vendorid", dfs["Vendor"], "businessentityid",
                       "PurchaseOrderHeader.vendorid → Vendor.businessentityid")
verifier_cle_etrangere(dfs["ProductVendor"], "businessentityid", dfs["Vendor"], "businessentityid",
                       "ProductVendor.businessentityid → Vendor.businessentityid")
verifier_cle_etrangere(dfs["ProductVendor"], "productid", dfs["Product"], "productid",
                       "ProductVendor.productid → Product.productid")
verifier_cle_etrangere(dfs["Product"], "productsubcategoryid", dfs["ProductSubcategory"], "productsubcategoryid",
                       "Product.productsubcategoryid → ProductSubcategory.productsubcategoryid")
verifier_cle_etrangere(dfs["ProductSubcategory"], "productcategoryid", dfs["ProductCategory"], "productcategoryid",
                       "ProductSubcategory.productcategoryid → ProductCategory.productcategoryid")
verifier_cle_etrangere(dfs["Customer"], "personid", dfs["Person"], "businessentityid",
                       "Customer.personid → Person.businessentityid")
verifier_cle_etrangere(dfs["SalesOrderHeader"], "customerid", dfs["Customer"], "customerid",
                       "SalesOrderHeader.customerid → Customer.customerid")
verifier_cle_etrangere(dfs["SalesOrderDetail"], "salesorderid", dfs["SalesOrderHeader"], "salesorderid",
                       "SalesOrderDetail.salesorderid → SalesOrderHeader.salesorderid")
verifier_cle_etrangere(dfs["ProductInventory"], "locationid", dfs["Location"], "locationid",
                       "ProductInventory.locationid → Location.locationid")

✅ PurchaseOrderDetail.productid → Product.productid : intégrité référentielle OK
✅ SalesOrderDetail.productid → Product.productid : intégrité référentielle OK
✅ PurchaseOrderHeader.vendorid → Vendor.businessentityid : intégrité référentielle OK
✅ ProductVendor.businessentityid → Vendor.businessentityid : intégrité référentielle OK
✅ ProductVendor.productid → Product.productid : intégrité référentielle OK
✅ Product.productsubcategoryid → ProductSubcategory.productsubcategoryid : intégrité référentielle OK
✅ ProductSubcategory.productcategoryid → ProductCategory.productcategoryid : intégrité référentielle OK
✅ Customer.personid → Person.businessentityid : intégrité référentielle OK
✅ SalesOrderHeader.customerid → Customer.customerid : intégrité référentielle OK
✅ SalesOrderDetail.salesorderid → SalesOrderHeader.salesorderid : intégrité référentielle OK
✅ ProductInventory.locationid → Location.locationid : intégrité référentielle OK


**Constat :** les 11 relations testées ne présentent **aucune valeur orpheline**. Le modèle relationnel peut être créé avec ses clés étrangères sans nettoyage préalable.

## 4. Cohérence logique

- **Dates** : la date de fin (expédition, échéance) ne doit pas précéder la date de commande. Les lignes où la date de fin est manquante sont ignorées (NULL ≠ incohérence).
- **Valeurs négatives** : quantités et prix doivent être positifs.

In [10]:
def verifier_dates_coherentes(df, col_debut, col_fin, nom_table):
    d1 = pd.to_datetime(df[col_debut], errors="coerce")
    d2 = pd.to_datetime(df[col_fin], errors="coerce")
    # Toute comparaison avec NaT renvoie False : les dates manquantes sont donc exclues
    incoherentes = df[d2 < d1]
    if len(incoherentes) > 0:
        print(f"⚠️ {nom_table} : {len(incoherentes)} lignes où {col_fin} < {col_debut}")
    else:
        print(f"✅ {nom_table} : dates cohérentes ({col_debut} <= {col_fin})")


def verifier_valeurs_negatives(df, colonne, nom_table):
    valeurs = pd.to_numeric(df[colonne], errors="coerce")
    negatives = df[valeurs < 0]
    if len(negatives) > 0:
        print(f"⚠️ {nom_table}.{colonne} : {len(negatives)} valeurs négatives")
    else:
        print(f"✅ {nom_table}.{colonne} : aucune valeur négative")

In [11]:
verifier_dates_coherentes(dfs["PurchaseOrderHeader"], "orderdate", "shipdate", "PurchaseOrderHeader")
verifier_dates_coherentes(dfs["SalesOrderHeader"], "orderdate", "shipdate", "SalesOrderHeader")
verifier_dates_coherentes(dfs["SalesOrderHeader"], "orderdate", "duedate", "SalesOrderHeader")

verifier_valeurs_negatives(dfs["PurchaseOrderDetail"], "orderqty", "PurchaseOrderDetail")
verifier_valeurs_negatives(dfs["SalesOrderDetail"], "orderqty", "SalesOrderDetail")
verifier_valeurs_negatives(dfs["Product"], "standardcost", "Product")
verifier_valeurs_negatives(dfs["Product"], "listprice", "Product")

✅ PurchaseOrderHeader : dates cohérentes (orderdate <= shipdate)
✅ SalesOrderHeader : dates cohérentes (orderdate <= shipdate)
✅ SalesOrderHeader : dates cohérentes (orderdate <= duedate)
✅ PurchaseOrderDetail.orderqty : aucune valeur négative
✅ SalesOrderDetail.orderqty : aucune valeur négative
✅ Product.standardcost : aucune valeur négative
✅ Product.listprice : aucune valeur négative


**Constat :** aucune incohérence de dates ni aucune valeur négative sur les quantités et les prix.

## 5. Diagnostics par axe d'analyse

### Fonction utilitaire : contrôle du `linetotal`

Les tables `PurchaseOrderDetail` et `SalesOrderDetail` contiennent bien une colonne `linetotal` (cf. types de données en section 1). On la **recalcule** pour vérifier sa cohérence :
- achats : `orderqty × unitprice`
- ventes : `orderqty × unitprice × (1 − unitpricediscount)`

In [12]:
def calculer_linetotal(df, col_qty, col_price, nom_table, col_discount=None, tolerance=0.01):
    if col_discount is not None:
        linetotal = df[col_qty] * df[col_price] * (1 - df[col_discount])
    else:
        linetotal = df[col_qty] * df[col_price]
    print(f"ℹ️ {nom_table} : linetotal recalculé, total = {linetotal.sum():,.2f}")

    # Comparaison avec la colonne existante (écart toléré : arrondis)
    if "linetotal" in df.columns:
        ecarts = (linetotal - df["linetotal"]).abs() > tolerance
        if ecarts.sum() > 0:
            print(f"⚠️ {nom_table} : {ecarts.sum()} lignes où linetotal diffère du recalcul")
        else:
            print(f"✅ {nom_table} : linetotal cohérent avec le recalcul")
    return linetotal

### 5.1 Performance fournisseurs

Indicateurs calculés avec la même logique que les mesures Power BI :
- **Délai Moyen Livraison (jours)** = moyenne, par commande d'achat, de `shipdate − orderdate` ;
- **Taux de Rejet** = `rejectedqty` total / `receivedqty` total (quantités rejetées rapportées aux quantités **reçues**) ;
- **Taux de Réception** = `receivedqty` total / `orderqty` total.

In [13]:
header = dfs["PurchaseOrderHeader"]
detail = dfs["PurchaseOrderDetail"]

# Délai de livraison
orderdate = pd.to_datetime(header["orderdate"], errors="coerce")
shipdate = pd.to_datetime(header["shipdate"], errors="coerce")
delai = (shipdate - orderdate).dt.days
print("Délai de livraison (jours) — statistiques :")
print(delai.describe().to_string())
print(f"\n⚠️ Commandes avec délai négatif : {(delai < 0).sum()}")
print(f"⚠️ Commandes sans date d'expédition (shipdate manquant) : {header['shipdate'].isnull().sum()}")

# Mesures Power BI « Taux de Rejet » et « Taux de Réception »
total_commande = detail["orderqty"].sum()
total_recu = detail["receivedqty"].sum()
total_rejete = detail["rejectedqty"].sum()
if total_recu > 0:
    print(f"\nTaux de rejet (rejeté / reçu)     : {total_rejete / total_recu:.2%}")
    print(f"Taux de réception (reçu / commandé) : {total_recu / total_commande:.2%}")
else:
    print("\n⚠️ Impossible de calculer le taux de rejet (receivedqty total = 0)")

calculer_linetotal(detail, "orderqty", "unitprice", "PurchaseOrderDetail");

Délai de livraison (jours) — statistiques :
count    4012.000000
mean        9.047856
std         0.873844
min         9.000000
25%         9.000000
50%         9.000000
75%         9.000000
max        25.000000

⚠️ Commandes avec délai négatif : 0
⚠️ Commandes sans date d'expédition (shipdate manquant) : 0

Taux de rejet (rejeté / reçu)     : 3.12%
Taux de réception (reçu / commandé) : 99.09%
ℹ️ PurchaseOrderDetail : linetotal recalculé, total = 63,791,994.84
✅ PurchaseOrderDetail : linetotal cohérent avec le recalcul


**Constat :**
- Délai de livraison quasi constant : **9 jours** pour la très grande majorité des commandes (max 25). Variabilité très faible → peu utile pour prédire les retards : presque toutes les commandes ont le même délai, le modèle n'aurait pas assez d'exemples de retard pour apprendre.
- Aucune commande sans date d'expédition ni avec délai négatif.
- Taux de rejet : **3,12 %** des quantités reçues (même valeur que la mesure Power BI « Taux de Rejet ») — indicateur exploitable par fournisseur.
- Taux de réception : **99,1 %** : une petite partie des quantités commandées n'est pas livrée.

### 5.2 Rotation des stocks

Distribution des quantités en stock, produits sans inventaire et **produits sous seuil**, calculés comme la mesure Power BI « Produits sous Seuil » :
- périmètre : produits finis (`finishedgoodsflag = 1`) ;
- stock : somme de `quantity` sur **tous les emplacements** ; un produit sans enregistrement d'inventaire a un stock de 0 ;
- seuil : point de commande (`reorderpoint`).

In [14]:
product = dfs["Product"]
inventory = dfs["ProductInventory"]

print("Distribution des quantités en stock :")
print(inventory["quantity"].describe().to_string())

produits_sans_stock = set(product["productid"]) - set(inventory["productid"])
print(f"\n⚠️ Produits sans enregistrement d'inventaire : {len(produits_sans_stock)}")

# Mesure Power BI « Produits sous Seuil »
stock_produit = inventory.groupby("productid")["quantity"].sum()
finis = product[product["finishedgoodsflag"] == 1].copy()
finis["stock"] = finis["productid"].map(stock_produit).fillna(0)   # pas d'inventaire = stock 0
finis["sans_inventaire"] = ~finis["productid"].isin(inventory["productid"])
sous_seuil = finis[finis["stock"] < finis["reorderpoint"]]

print(f"\nProduits finis : {len(finis)}")
print(f"⚠️ Produits sous seuil (stock < reorderpoint) : {len(sous_seuil)}")
print(f"   dont sans aucun enregistrement d'inventaire : {sous_seuil['sans_inventaire'].sum()}")
print(f"   dont avec un stock existant mais insuffisant : {(~sous_seuil['sans_inventaire']).sum()}")
print(sous_seuil.loc[~sous_seuil["sans_inventaire"], ["productid", "name", "stock", "reorderpoint"]].to_string(index=False))

# Les produits finis sans inventaire sont-ils vendus ?
vendus = set(dfs["SalesOrderDetail"]["productid"])
sans_inv = sous_seuil[sous_seuil["sans_inventaire"]]
print(f"\nProduits sous seuil sans inventaire et pourtant vendus : {sans_inv['productid'].isin(vendus).sum()} / {len(sans_inv)}")

verifier_valeurs_negatives(inventory, "quantity", "ProductInventory")

Distribution des quantités en stock :
count    1069.00000
mean      314.28812
std       189.85056
min         0.00000
25%       140.00000
50%       299.00000
75%       481.00000
max       924.00000

⚠️ Produits sans enregistrement d'inventaire : 72

Produits finis : 295
⚠️ Produits sous seuil (stock < reorderpoint) : 77
   dont sans aucun enregistrement d'inventaire : 72
   dont avec un stock existant mais insuffisant : 5
 productid                           name  stock  reorderpoint
       853              Women's Tights, M    0.0             3
       859          Half-Finger Gloves, M    0.0             3
       876            Hitch Rack - 4-Bike    0.0             3
       882 Short-Sleeve Classic Jersey, M    0.0             3
       910        HL Mountain Seat/Saddle  355.0           375



Produits sous seuil sans inventaire et pourtant vendus : 56 / 72
✅ ProductInventory.quantity : aucune valeur négative


**Constat :**
- **72 produits** n'ont aucun enregistrement d'inventaire.
- **77 produits finis sur 295** sont sous leur point de commande (même valeur que la mesure Power BI « Produits sous Seuil ») :
  - **72** n'ont aucun enregistrement d'inventaire (stock considéré à 0), dont 56 sont pourtant vendus → rupture probable ;
  - **5** ont un stock réel mais insuffisant (4 à 0 unité, 1 selle à 355 pour un seuil de 375).
- Aucune quantité négative.

*Remarque : `ProductInventory` contient une ligne par emplacement (entrepôt et ateliers). Le stock est donc additionné par produit avant la comparaison au seuil, comme dans Power BI.*

### 5.3 Marge commerciale

Calcul identique aux mesures Power BI :
- **Chiffre d'affaires net** = Σ `orderqty × unitprice × (1 − unitpricediscount)` ;
- **Coût des ventes** = Σ `orderqty × standardcost` ;
- **Marge commerciale** = CA net − coût des ventes ; **Taux de marge** = marge / CA net (moyenne **pondérée par le CA**).

La marge est aussi calculée par canal (mesure Power BI « Canal » : en ligne / hors ligne) et ligne à ligne pour repérer les ventes à perte.

In [15]:
sales = dfs["SalesOrderDetail"]
soh_canal = dfs["SalesOrderHeader"][["salesorderid", "onlineorderflag"]]

merged_marge = (sales.merge(product, on="productid", how="inner")
                     .merge(soh_canal, on="salesorderid", how="left"))
merged_marge["ca_net"] = merged_marge["orderqty"] * merged_marge["unitprice"] * (1 - merged_marge["unitpricediscount"])
merged_marge["cout_ventes"] = merged_marge["orderqty"] * merged_marge["standardcost"]
merged_marge["marge"] = merged_marge["ca_net"] - merged_marge["cout_ventes"]
merged_marge["canal"] = merged_marge["onlineorderflag"].map({1: "En ligne", 0: "Hors ligne"})

# Mesures globales
ca_net = merged_marge["ca_net"].sum()
cout = merged_marge["cout_ventes"].sum()
print(f"Chiffre d'affaires net : {ca_net:,.2f}")
print(f"Coût des ventes        : {cout:,.2f}")
print(f"Marge commerciale      : {ca_net - cout:,.2f}")
print(f"Taux de marge          : {(ca_net - cout) / ca_net:.1%}")

# Par canal
par_canal = merged_marge.groupby("canal")[["ca_net", "cout_ventes", "marge"]].sum()
par_canal["taux_marge"] = par_canal["marge"] / par_canal["ca_net"]
par_canal["part_ca"] = par_canal["ca_net"] / ca_net
print("\nPar canal :")
print(par_canal.round(3).to_string())

# Lignes vendues à perte (marge de la ligne < 0)
negatives = merged_marge[merged_marge["marge"] < 0]
print(f"\n⚠️ Lignes avec marge négative (vente à perte) : {len(negatives)} ({len(negatives) / len(merged_marge):.1%} des lignes)")

calculer_linetotal(sales, "orderqty", "unitprice", "SalesOrderDetail", col_discount="unitpricediscount");

Chiffre d'affaires net : 109,846,381.40
Coût des ventes        : 100,474,477.77
Marge commerciale      : 9,371,903.63
Taux de marge          : 8.5%

Par canal :
                  ca_net   cout_ventes         marge  taux_marge  part_ca
canal                                                                    
En ligne    2.935868e+07  1.767073e+07  1.168794e+07       0.398    0.267
Hors ligne  8.048770e+07  8.280374e+07 -2.316039e+06      -0.029    0.733

⚠️ Lignes avec marge négative (vente à perte) : 29161 (24.0% des lignes)
ℹ️ SalesOrderDetail : linetotal recalculé, total = 109,846,381.40
✅ SalesOrderDetail : linetotal cohérent avec le recalcul


**Constat :**
- **Taux de marge global : 8,5 %** (CA net ≈ 109,8 M, coût des ventes ≈ 100,5 M), identique à la mesure Power BI « Taux de marge ».
- **Écart très fort entre canaux** :
  - **en ligne : 39,8 %** de marge, 27 % du CA ;
  - **hors ligne (revendeurs) : −2,9 %**, mais **73 % du CA**. Le canal revendeur, majoritaire, fait chuter la marge globale.
- **29 161 lignes** sont vendues à perte (24 % des lignes) : analysé ci-dessous (section 5.4).

### 5.4 Origine des lignes en marge négative

Hypothèses testées : (1) les remises (`unitpricediscount`), (2) le canal de vente (`onlineorderflag` : 1 = en ligne, 0 = revendeur), (3) le niveau du prix de vente par rapport au prix catalogue (`listprice`).

In [16]:
negatives = merged_marge[merged_marge["marge"] < 0]

# (1) Remises
print(f"Lignes en marge négative : {len(negatives)}")
print(f"  dont avec remise (unitpricediscount > 0) : {(negatives['unitpricediscount'] > 0).sum()}")

# (2) Canal de vente
print("\nLignes en marge négative par canal :")
print(pd.crosstab(merged_marge["canal"], merged_marge["marge"] < 0,
                  rownames=["canal"], colnames=["marge négative"], margins=True))

# (3) Prix de vente / prix catalogue, par canal
merged_marge["ratio_prix_catalogue"] = (merged_marge["unitprice"] / merged_marge["listprice"]).round(2)
print("\nPrix de vente / prix catalogue (valeurs les plus fréquentes) :")
for canal in ["En ligne", "Hors ligne"]:
    freq = merged_marge.loc[merged_marge["canal"] == canal, "ratio_prix_catalogue"].value_counts().head(3)
    print(f"  {canal} : {freq.to_dict()}")

# Coût standard / prix catalogue des produits vendus à perte
ratio_cout = negatives["standardcost"] / negatives["listprice"]
print(f"\nProduits vendus à perte : {negatives['productid'].nunique()} — coût standard = "
      f"{ratio_cout.min():.0%} à {ratio_cout.max():.0%} du prix catalogue (médiane {ratio_cout.median():.0%})")

# Répartition par catégorie (perte = −marge)
sub = dfs["ProductSubcategory"][["productsubcategoryid", "productcategoryid"]]
cat = dfs["ProductCategory"][["productcategoryid", "name"]].rename(columns={"name": "categorie"})
neg_cat = negatives.merge(sub, on="productsubcategoryid", how="left").merge(cat, on="productcategoryid", how="left")
print("\nPar catégorie :")
print(neg_cat.groupby("categorie").agg(lignes=("productid", "size"), perte=("marge", lambda m: -m.sum()))
      .sort_values("perte", ascending=False).round(0).to_string())
print(f"\nPerte totale sur les lignes vendues à perte : {-negatives['marge'].sum():,.0f}")

Lignes en marge négative : 29161
  dont avec remise (unitpricediscount > 0) : 1824

Lignes en marge négative par canal :
marge négative  False   True     All
canal                               
En ligne        60398      0   60398
Hors ligne      31758  29161   60919
All             92156  29161  121317

Prix de vente / prix catalogue (valeurs les plus fréquentes) :
  En ligne : {1.0: 57949, 0.89: 2449}
  Hors ligne : {0.6: 43553, 0.54: 5650, 0.58: 5424}

Produits vendus à perte : 121 — coût standard = 41% à 77% du prix catalogue (médiane 62%)

Par catégorie :
            lignes      perte
categorie                    
Bikes        19340  4595091.0
Components    4769   238751.0
Clothing      5052   200546.0

Perte totale sur les lignes vendues à perte : 5,034,388


**Constat :**
- **Les remises ne sont pas la cause** : seules 1 824 des 29 161 lignes ont une remise.
- **Le canal explique tout** : 100 % des lignes vendues à perte sont des **ventes hors ligne (revendeurs)**, soit environ 48 % des lignes revendeurs. Aucune vente en ligne n'est à perte.
- **Mécanisme** : en ligne, le prix de vente = prix catalogue ; les revendeurs paient environ **60 % du prix catalogue**. Les 121 produits concernés ont un coût standard compris entre 41 % et 77 % du prix catalogue (médiane 62 %) : avec une remise revendeur de 40 %, le prix passe sous le coût.
- **Les vélos concentrent l'essentiel** : 19 340 lignes et environ 4,6 M de perte sur les 5,0 M au total, devant Components et Clothing.

**Décision :** la marge est analysée **par canal** (en ligne vs hors ligne) dans Power BI ; le levier principal est la politique tarifaire revendeur sur les vélos, et non les remises ponctuelles.

## 6. Analyse approfondie : prix d'achat réel vs prix standard (`ProductVendor`)

La table `ProductVendor` contient pour chaque couple produit–fournisseur :
- `standardprice` : prix d'achat standard (contractuel)
- `lastreceiptcost` : coût réel lors de la dernière réception

**Question :** l'écart entre ces deux prix varie-t-il selon les fournisseurs ? Si oui, c'est un indicateur de performance fournisseur exploitable.

### 6.1 Ratio `lastreceiptcost / standardprice`

In [17]:
pv = dfs["ProductVendor"].copy()
pv["ratio"] = pv["lastreceiptcost"] / pv["standardprice"]

print(pv["ratio"].round(4).value_counts())

ratio
1.0500    406
1.0000      6
1.0392      4
0.9564      3
1.0457      3
1.0110      3
1.0237      3
1.0287      3
0.9876      2
1.0183      2
0.9893      2
0.7867      2
1.0968      1
1.0897      1
0.9424      1
1.0565      1
1.0278      1
1.0184      1
0.9252      1
0.9513      1
1.2129      1
0.9706      1
0.9581      1
1.2822      1
0.3055      1
1.1784      1
0.9636      1
1.1200      1
1.0750      1
0.7058      1
0.9900      1
1.2135      1
1.0821      1
Name: count, dtype: int64


**Constat :** 406 lignes sur 460 ont un ratio exactement égal à **1,05**. Il s'agit d'une règle de génération des données (`lastreceiptcost = standardprice × 1,05`) et non d'une variation réelle des prix. Ces lignes n'apportent donc aucune information sur la performance des fournisseurs.

### 6.2 Isolement des exceptions

In [18]:
exceptions = pv[pv["ratio"].round(4) != 1.05]
print(f"Nombre d'exceptions : {len(exceptions)}")

exceptions[["productid", "businessentityid", "standardprice", "lastreceiptcost", "ratio"]].sort_values("ratio")

Nombre d'exceptions : 54


,productid,businessentityid,standardprice,lastreceiptcost,ratio
373,847,1518,14.50,4.4300,0.305517
402,876,1546,48.00,33.8800,0.705833
400,874,1574,3.00,2.3600,0.786667
401,875,1574,3.00,2.3600,0.786667
367,841,1636,26.75,24.7500,0.925234
361,711,1520,13.88,13.0800,0.942363
368,842,1518,54.20,51.5600,0.951292
378,852,1594,32.35,30.9400,0.956414
380,854,1594,32.35,30.9400,0.956414
379,853,1594,32.35,30.9400,0.956414


**Constat :** 54 lignes s'écartent de la règle. Les `productid` se situent tous dans la plage 707–878 et les mêmes fournisseurs reviennent régulièrement.

### 6.3 Type de produit concerné

In [19]:
cols_produit = ["productid", "name", "makeflag", "finishedgoodsflag"]
exc = exceptions.merge(product[cols_produit], on="productid", how="left")

exc[["makeflag", "finishedgoodsflag"]].value_counts()

makeflag  finishedgoodsflag
0         1                    54
Name: count, dtype: int64

**Constat :** les 54 exceptions concernent **exclusivement des produits finis achetés pour la revente** (`makeflag = 0`, `finishedgoodsflag = 1`). Les composants de production suivent tous la règle fixe de 1,05.

### 6.4 Vérification inverse : tous les produits de revente sont-ils des exceptions ?

Si certains produits de revente ont aussi un ratio de 1,05, cela signifie que le type de produit ne suffit pas à expliquer l'écart : le fournisseur joue aussi un rôle.

In [20]:
pv_full = pv.merge(product[["productid", "makeflag", "finishedgoodsflag"]], on="productid", how="left")
revente = pv_full[(pv_full["makeflag"] == 0) & (pv_full["finishedgoodsflag"] == 1)]

print(f"Nombre de lignes produits de revente : {len(revente)}")
print((revente["ratio"].round(4) != 1.05).value_counts().rename({True: "exception", False: "ratio = 1,05"}))

Nombre de lignes produits de revente : 103
ratio
exception       54
ratio = 1,05    49
Name: count, dtype: int64


**Interprétation :** sur 103 lignes de produits de revente, **54 sont des exceptions et 49 suivent la règle de 1,05**. Le type de produit est donc une **condition nécessaire mais pas suffisante** : l'écart de prix dépend aussi du fournisseur.

### 6.5 Écart moyen par fournisseur

In [21]:
vendor = dfs["Vendor"][["businessentityid", "name"]].rename(columns={"name": "fournisseur"})

par_fournisseur = (
    exc.groupby("businessentityid")["ratio"]
       .agg(nb_lignes="count", ratio_moyen="mean")
       .reset_index()
       .merge(vendor, on="businessentityid", how="left")
       .sort_values("ratio_moyen")
)
par_fournisseur["ecart_pct"] = ((par_fournisseur["ratio_moyen"] - 1) * 100).round(1)
par_fournisseur

,businessentityid,nb_lignes,ratio_moyen,fournisseur,ecart_pct
3,1574,4,0.939963,Jeff's Sporting Goods,-6.0
1,1520,3,0.972551,G & K Bicycle Corp.,-2.7
4,1594,9,0.979061,Fitness Association,-2.1
0,1518,7,0.979873,International Trek Center,-2.0
2,1546,9,1.020712,Green Lake Bike Company,2.1
5,1636,10,1.022140,Integrated Sport Products,2.2
6,1676,12,1.027268,Team Athletic Co.,2.7


**Constat :** les exceptions ne concernent que **7 fournisseurs** :
- **coût réel inférieur au prix standard** : Jeff's Sporting Goods (1574, −6,0 %), G & K Bicycle Corp. (1520, −2,7 %), Fitness Association (1594, −2,1 %)
- **coût réel supérieur au prix standard** : Green Lake Bike Company (1546, +2,1 %), Integrated Sport Products (1636, +2,2 %), Team Athletic Co. (1676, +2,7 %)
- International Trek Center (1518, −2,0 %) : moyenne faussée par une valeur aberrante, voir ci-dessous.

### 6.6 Valeur aberrante

In [22]:
exc[exc["ratio"] < 0.5][["productid", "name", "businessentityid", "standardprice", "lastreceiptcost", "ratio"]]

,productid,name,businessentityid,standardprice,lastreceiptcost,ratio
16,847,Headlights - Dual-Beam,1518,14.5,4.43,0.305517


**Constat :** le produit 847 (fournisseur 1518) présente un coût réel égal à **31 %** du prix standard (14,50 → 4,43). Il s'agit probablement d'une erreur de saisie ou d'une remise exceptionnelle. Cette ligne est signalée comme **valeur aberrante** et doit être exclue des calculs de moyenne.

In [23]:
# Écart moyen par fournisseur, hors valeur aberrante
exc_sans_outlier = exc[exc["ratio"] >= 0.5]
exc_sans_outlier.groupby("businessentityid")["ratio"].agg(["count", "mean"]).sort_values("mean")

,count,mean
businessentityid,,
1574,4,0.939963
1520,3,0.972551
1594,9,0.979061
1546,9,1.020712
1636,10,1.022140
1676,12,1.027268
1518,6,1.092266


**Constat :** une fois la valeur aberrante exclue, International Trek Center (1518) passe de −2,0 % à **+9,2 %** : c'est en réalité le fournisseur dont le coût réel dépasse le plus le prix standard. Cela confirme l'importance de traiter les valeurs aberrantes avant de comparer les fournisseurs.

## 7. Synthèse

| Thème | Constat | Décision |
|---|---|---|
| Valeurs manquantes | 23 colonnes concernées, majoritairement structurelles ; 2 colonnes vides à 100 % | Conservées NULL ; `shelf` → « N/A », `onorderqty` → 0 ; `discontinueddate` et `comment` supprimables |
| Zéros « déguisés » | 200 produits (39,7 %) à `standardcost` = `listprice` = 0, tous non vendables | Traités comme « non applicable », exclus des calculs de prix et de marge |
| Doublons | Aucune ligne dupliquée sur les 15 tables | Aucune action |
| Clés primaires | 15 tables : uniques et non nulles | Aucune action |
| Intégrité référentielle | 11 relations sans valeur orpheline | Aucune action |
| Cohérence logique | Dates cohérentes, aucune valeur négative | Aucune action |
| Délai fournisseur | Quasi constant (9 jours), max 25 | Faible variabilité à prendre en compte pour le ML |
| Taux de rejet | 3,12 % des quantités reçues (mesure Power BI) | Indicateur retenu par fournisseur |
| Stock sous seuil | **77 / 295 produits finis** sous le point de commande (mesure Power BI) : 72 sans inventaire, 5 en stock insuffisant | Stock additionné par produit, absence d'inventaire = stock 0 |
| Marge | Taux global **8,5 %** (pondéré par le CA) : 39,8 % en ligne, −2,9 % hors ligne (73 % du CA) ; 29 161 lignes à perte, 100 % revendeurs, surtout des vélos | Analyse de la marge par canal ; levier = tarif revendeur |
| `ProductVendor` — ratio de prix | 406/460 lignes à 1,05 (règle de génération) | Non exploitable comme indicateur |
| `ProductVendor` — exceptions | 54 lignes, uniquement produits de revente, 7 fournisseurs ; 49 produits de revente restent à 1,05 | Indicateur d'écart de prix retenu pour ces 7 fournisseurs |
| `ProductVendor` — valeur aberrante | Produit 847 / fournisseur 1518, ratio 0,31 | Exclue des moyennes ; 1518 passe à +9,2 % |